# Titanic
## Score: .76555

In [18]:
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ROOT = Path.cwd()
DATA = ROOT / "titanic"
assert (DATA / "train.csv").exists(), "Set cwd to project root (folder containing titanic/train.csv)"

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [19]:
def extract_title(name: str) -> str:
    m = re.search(r",\s*([^\.]+)\.", name)
    return m.group(1).strip() if m else "Unknown"


def cabin_deck(cabin) -> str:
    if pd.isna(cabin) or not str(cabin).strip():
        return "U"
    c = str(cabin).strip()[0]
    return c if c.isalpha() else "U"


def add_features(df: pd.DataFrame, ticket_vc: pd.Series | None = None) -> pd.DataFrame:
    out = df.copy()
    out["Title"] = out["Name"].map(extract_title)
    rare = {
        "Lady",
        "Countess",
        "Capt",
        "Col",
        "Don",
        "Dr",
        "Major",
        "Rev",
        "Sir",
        "Jonkheer",
        "Dona",
    }
    out["Title"] = out["Title"].replace(
        {
            "Mlle": "Miss",
            "Ms": "Miss",
            "Mme": "Mrs",
        }
    )
    out.loc[out["Title"].isin(rare), "Title"] = "Rare"
    out["FamilySize"] = out["SibSp"] + out["Parch"] + 1
    out["IsAlone"] = (out["FamilySize"] == 1).astype(int)
    out["HasCabin"] = out["Cabin"].notna().astype(int)
    out["Deck"] = out["Cabin"].map(cabin_deck)
    out["FarePerPerson"] = out["Fare"] / out["FamilySize"].clip(lower=1)
    out["LogFare"] = np.log1p(out["Fare"])
    out["IsChild"] = np.where(out["Age"].notna(), (out["Age"] < 16).astype(int), 0)
    if ticket_vc is None:
        out["TicketGroupSize"] = out.groupby("Ticket")["Ticket"].transform("count")
    else:
        out["TicketGroupSize"] = out["Ticket"].map(ticket_vc).fillna(1).astype(int)
    return out

In [20]:
def build_pipeline() -> Pipeline:
    numeric = [
        "Pclass",
        "Age",
        "SibSp",
        "Parch",
        "Fare",
        "FamilySize",
        "IsAlone",
        "HasCabin",
        "FarePerPerson",
        "LogFare",
        "IsChild",
        "TicketGroupSize",
    ]
    categorical = ["Sex", "Embarked", "Title", "Deck"]

    pre = ColumnTransformer(
        [
            ("num", SimpleImputer(strategy="median"), numeric),
            (
                "cat",
                Pipeline(
                    [
                        ("impute", SimpleImputer(strategy="most_frequent")),
                        ("oh", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                categorical,
            ),
        ]
    )

    clf = RandomForestClassifier(
        n_estimators=400,
        max_depth=8,
        min_samples_leaf=3,
        min_samples_split=6,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1,
    )
    return Pipeline([("prep", pre), ("model", clf)])

In [21]:
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")

ticket_vc = train["Ticket"].value_counts()
train_x = add_features(train.drop(columns=["Survived"]), ticket_vc)
train_y = train["Survived"]
test_x = add_features(test, ticket_vc)

pipe = build_pipeline()
cv_scores = cross_val_score(pipe, train_x, train_y, cv=CV, scoring="accuracy", n_jobs=-1)
print(f"RF CV accuracy @ prob>=0.5: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")

oof_proba_cv = cross_val_predict(pipe, train_x, train_y, cv=CV, method="predict_proba", n_jobs=-1)[:, 1]
thresh_grid = np.linspace(0.30, 0.65, 72)
best_t, best_oob = 0.5, -1.0
for t in thresh_grid:
    acc = accuracy_score(train_y, (oof_proba_cv >= t).astype(int))
    if acc > best_oob:
        best_oob, best_t = acc, t
print(f"OOF accuracy after threshold sweep: {best_oob:.4f} (best threshold={best_t:.3f})")

pipe.fit(train_x, train_y)
test_proba = pipe.predict_proba(test_x)[:, 1]
pred = (test_proba >= best_t).astype(int)
sub = pd.DataFrame({"PassengerId": test["PassengerId"], "Survived": pred.astype(int)})
out_path = ROOT / "submission.csv"
sub.to_csv(out_path, index=False)
print(f"Wrote {out_path} ({len(sub)} rows)")
sub.head(10)

RF CV accuracy @ prob>=0.5: 0.8384 +/- 0.0099
OOF accuracy after threshold sweep: 0.8395 (best threshold=0.448)
Wrote c:\Users\ol1v3_7dwns5u\OneDrive\Desktop\ACTIVE\Titanic\submission.csv (418 rows)


,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1
5,897,0
6,898,1
7,899,0
8,900,1
9,901,0


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import cross_val_predict

oof_proba = cross_val_predict(pipe, train_x, train_y, cv=CV, method="predict_proba", n_jobs=-1)[:, 1]
oof_default = (oof_proba >= 0.5).astype(int)
thresh_grid = np.linspace(0.30, 0.65, 72)
best_t_eval, best_acc_eval = 0.5, -1.0
for t in thresh_grid:
    a = accuracy_score(train_y, (oof_proba >= t).astype(int))
    if a > best_acc_eval:
        best_acc_eval, best_t_eval = a, t
oof_tuned = (oof_proba >= best_t_eval).astype(int)

print("=== OOF metrics (same CV folds as training cell) ===")
print(f"Threshold sweep on OOF proba: best={best_t_eval:.3f}, OOF acc={best_acc_eval:.4f} (vs 0.5 -> {accuracy_score(train_y, oof_default):.4f})")

cm_d = confusion_matrix(train_y, oof_default)
cm_t = confusion_matrix(train_y, oof_tuned)
tn, fp, fn, tp = cm_d.ravel()
print(f"\nAt 0.5: FN (missed survivors)={fn}, FP (false survivors)={fp}  <- model skews toward death when FN>>FP")
print("At 0.5:\n", classification_report(train_y, oof_default, digits=4))
print(f"At tuned threshold {best_t_eval:.3f}:\n", classification_report(train_y, oof_tuned, digits=4))
print("Confusion @0.5 [TN FP; FN TP]:\n", cm_d, "\nConfusion @tuned:\n", cm_t)
print(f"ROC-AUC (OOF): {roc_auc_score(train_y, oof_proba):.4f}")

fold_acc = []
for _, te_idx in CV.split(train_x, train_y):
    fold_acc.append(accuracy_score(train_y.iloc[te_idx], oof_tuned[te_idx]))
print(f"\nPer-fold OOF accuracy (tuned preds): {[round(x, 4) for x in fold_acc]}  std={np.std(fold_acc):.4f}")

train_aug = train.assign(oof_wrong=oof_tuned != train_y.values)
print("\nOOF error rate by Sex:", train_aug.groupby("Sex")["oof_wrong"].mean().round(4).to_dict())
print("OOF error rate by Pclass:", train_aug.groupby("Pclass")["oof_wrong"].mean().round(4).to_dict())

maj = int(train_y.mode().iloc[0])
base = accuracy_score(train_y, np.full(len(train_y), maj))
print(f"\nMajority-class baseline accuracy on train: {base:.4f}")

prep = pipe.named_steps["prep"]
feat_names = prep.get_feature_names_out()
imp = pd.Series(pipe.named_steps["model"].feature_importances_, index=feat_names).sort_values(ascending=False)
print("\nTop feature importances (full-train fit):")
print(imp.head(15).to_string())

=== OOF metrics (same CV folds as training cell) ===
Threshold sweep on OOF proba: best=0.448, OOF acc=0.8395 (vs 0.5 -> 0.8384)

At 0.5: FN (missed survivors)=90, FP (false survivors)=54  <- model skews toward death when FN>>FP
At 0.5:
               precision    recall  f1-score   support

           0     0.8462    0.9016    0.8730       549
           1     0.8235    0.7368    0.7778       342

    accuracy                         0.8384       891
   macro avg     0.8348    0.8192    0.8254       891
weighted avg     0.8375    0.8384    0.8365       891

At tuned threshold 0.448:
               precision    recall  f1-score   support

           0     0.8612    0.8816    0.8713       549
           1     0.8024    0.7719    0.7869       342

    accuracy                         0.8395       891
   macro avg     0.8318    0.8268    0.8291       891
weighted avg     0.8386    0.8395    0.8389       891

Confusion @0.5 [TN FP; FN TP]:
 [[495  54]
 [ 90 252]] 
Confusion @tuned:
 [[484 